In [ ]:
from flask import Flask, render_template_string, request, session, redirect, url_for
import random
import os
import json

app = Flask(__name__)
app.secret_key = 'secret-key'

LEADERBOARD_FILE = "leaderboard.json"

words = [
    {"word": "meticulous", "sentence": "She kept meticulous records.", "level": 2},
    {"word": "aberration", "sentence": "The result was an aberration.", "level": 3},
    {"word": "candid", "sentence": "He gave a candid answer.", "level": 1}
]

def get_word(level):
    filtered = [w for w in words if w["level"] == level]
    return random.choice(filtered)

# Leaderboard helpers
def load_leaderboard():
    if not os.path.exists(LEADERBOARD_FILE):
        return []
    try:
        with open(LEADERBOARD_FILE, "r") as f:
            return json.load(f)
    except:
        return []

def save_leaderboard(data):
    with open(LEADERBOARD_FILE, "w") as f:
        json.dump(data, f)

# Title system
def get_title(score):
    if score >= 10:
        return "🏆 Champion"
    elif score >= 5:
        return "🥇 Finalist"
    elif score >= 3:
        return "🥈 Competitor"
    else:
        return "🟡 Participant"

HTML = """
<!DOCTYPE html>
<html>
<head>
<title>AI Spelling Bee</title>
<style>
body {font-family: Arial; background: linear-gradient(135deg,#0f172a,#1e3a8a); color:white; text-align:center;}
.header {background:#1e40af; padding:20px; font-size:36px; color:#facc15;}
.container {background:white; color:black; margin:40px auto; padding:30px; width:60%; border-radius:12px;}
.word {font-size:28px; color:#f59e0b;}
button {padding:10px; margin:8px; border-radius:8px; border:none; cursor:pointer;}
.btn-yellow {background:#facc15;}
.btn-dark {background:#1f2937; color:white;}
.stats {margin-top:20px;}
.timer {font-size:22px; color:red;}
.leaderboard {margin-top:20px; text-align:left;}
input {padding:10px; margin:10px; width:60%;}
</style>
</head>
<body>
<div class="header">🐝 AI SPELLING BEE</div>
<div class="container">

{% if not player_name %}
    <h2>Enter Your Name</h2>
    <form method="post">
        <input type="text" name="player_name" required placeholder="Your name">
        <button class="btn-yellow">Start Game</button>
    </form>
{% else %}

<h2>{{player_name}} | Round {{round}} | ❤️ {{lives}}</h2>

<div class="timer">⏱ Time Left: <span id="time">10</span>s</div>

<p>Listen to the word:</p>
<div class="word">🔊 Click Hear Word</div>

<button class="btn-dark" onclick="speakWord()">🔊 Hear Word ({{repeats}} left)</button>
<button class="btn-dark" onclick="speakSentence()">📘 Sentence</button>

<br>
<input type="text" id="answer" placeholder="Type or speak">
<br>
<button class="btn-yellow" onclick="startListening()">🎤 Speak</button>

<form method="post" onsubmit="syncAnswer()">
<input type="hidden" id="hiddenAnswer" name="answer">
<button class="btn-dark">Submit</button>
</form>

<div>{{message}}</div>

<div class="stats">
<p>Score: {{score}}</p>
<p>Accuracy: {{accuracy}}%</p>
<p>Attempts: {{attempts}}</p>
</div>

<div class="leaderboard">
<h3>🏆 Leaderboard</h3>
<ul>
{% for entry in leaderboard %}
<li>{{entry.name}} - {{entry.score}} ({{entry.title}})</li>
{% endfor %}
</ul>
</div>

{% endif %}
</div>

<script>
let timeLeft = 10;
let timer = setInterval(() => {
    let t = document.getElementById("time");
    if(!t) return;
    timeLeft--;
    t.innerText = timeLeft;
    if (timeLeft <= 0) {
        clearInterval(timer);
        document.querySelector("form[onsubmit]")?.submit();
    }
}, 1000);

function speak(text){speechSynthesis.speak(new SpeechSynthesisUtterance(text));}
function speakWord(){fetch('/repeat'); speak("Your word is {{word}}");}
function speakSentence(){speak("{{sentence}}");}

function startListening(){
const r=new(window.SpeechRecognition||window.webkitSpeechRecognition)();
r.start();
r.onresult=e=>{document.getElementById('answer').value=e.results[0][0].transcript.replace(/ /g,'').toLowerCase();};
}

function syncAnswer(){
document.getElementById('hiddenAnswer').value=document.getElementById('answer').value;
}
</script>
</body>
</html>
"""

@app.route('/', methods=['GET','POST'])
def index():
    # Name setup
    if 'player_name' not in session:
        if request.method == 'POST' and request.form.get('player_name'):
            session['player_name'] = request.form.get('player_name')
        else:
            return render_template_string(HTML, player_name=None)

    if 'round' not in session:
        session['round']=1
        session['lives']=2
        session['word']=get_word(1)
        session['correct']=0
        session['attempts']=0
        session['repeats']=2

    message=""

    if request.method=='POST' and request.form.get('answer') is not None:
        answer=request.form.get('answer','').lower()
        correct=session['word']['word']
        session['attempts']+=1

        if answer==correct:
            session['correct']+=1
            message="Correct!"
            session['round']+=1
            session['word']=get_word(min(session['round'],3))
            session['repeats']=2
        else:
            session['lives']-=1
            message=f"Incorrect. Word was {correct}"

    if session['lives']<=0:
        leaderboard = load_leaderboard()
        score = session['correct']
        title = get_title(score)
        leaderboard.append({"name": session['player_name'], "score": score, "title": title})
        leaderboard = sorted(leaderboard, key=lambda x: x['score'], reverse=True)[:5]
        save_leaderboard(leaderboard)

        return f"<h1 style='color:white;text-align:center;'>🏆 Game Over</h1><p style='text-align:center;color:white;'>Player: {session['player_name']}<br>Score: {score}<br>Title: {title}</p><div style='text-align:center;'><a href='/reset'>Play Again</a></div>"

    acc=int((session['correct']/session['attempts'])*100) if session['attempts'] else 0

    return render_template_string(HTML,
        player_name=session['player_name'],
        round=session['round'], lives=session['lives'], message=message,
        word=session['word']['word'], sentence=session['word']['sentence'],
        score=session['correct'], attempts=session['attempts'], accuracy=acc,
        leaderboard=load_leaderboard(), repeats=session['repeats'])

@app.route('/repeat')
def repeat():
    if session.get('repeats',0)>0:
        session['repeats']-=1
    return ('',204)

@app.route('/reset')
def reset():
    session.clear()
    return redirect(url_for('index'))

if __name__=='__main__':
    port=int(os.environ.get("PORT",10000))
    app.run(host='0.0.0.0', port=port)
